# Task Data Analysis Notebook

This notebook loads task run stats from `./.pm` and gives a quick view of completion rates, validation behavior, timings, and common failure patterns.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
plt.style.use("ggplot")

In [ ]:
candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "./.pm",
    Path.cwd().parent / "./.pm",
]

stats_dir = None
stat_files = []
for d in candidate_dirs:
    if d.is_dir():
        files = sorted(d.glob("*-stats.json"))
        if files:
            stats_dir = d
            stat_files = files
            break

print(f"Working directory: {Path.cwd()}")
print(f"Using stats directory: {stats_dir}")
print(f"Found {len(stat_files)} stats file(s).")
stat_files

In [ ]:
datasets = []
for file in stat_files:
    with file.open() as f:
        data = json.load(f)
    data["_file"] = file.name
    datasets.append(data)

print(f"Loaded {len(datasets)} task dataset(s).")

In [ ]:
summary_rows = []
run_rows = []
log_rows = []

for ds in datasets:
    task_name = ds.get("task_name")
    file_name = ds.get("_file")

    summary = ds.get("summary", {}).copy()
    summary.update({"task_name": task_name, "file": file_name})
    summary_rows.append(summary)

    for run in ds.get("runs", []):
        row = run.copy()
        row.update({"task_name": task_name, "file": file_name})
        run_rows.append(row)

        for log in run.get("logs", []):
            lrow = log.copy()
            lrow.update({
                "task_name": task_name,
                "file": file_name,
                "run_id": run.get("run_id"),
                "run_completed": run.get("completed"),
            })
            log_rows.append(lrow)

summary_df = pd.DataFrame(summary_rows)
runs_df = pd.DataFrame(run_rows)
logs_df = pd.DataFrame(log_rows)

for col in ["updated_at", "first_run_started_at", "last_run_started_at", "last_run_ended_at"]:
    if col in summary_df.columns:
        summary_df[col] = pd.to_datetime(summary_df[col], errors="coerce")

for col in ["started_at", "ended_at"]:
    if col in runs_df.columns:
        runs_df[col] = pd.to_datetime(runs_df[col], errors="coerce")

if "timestamp" in logs_df.columns:
    logs_df["timestamp"] = pd.to_datetime(logs_df["timestamp"], errors="coerce")

summary_df.shape, runs_df.shape, logs_df.shape

## Task-level summary

In [ ]:
summary_cols = [
    "task_name",
    "total_runs",
    "completed_runs",
    "incomplete_runs",
    "average_duration_ms",
    "validation_attempts",
    "validation_successes",
    "validation_failures",
    "total_user_actions",
]

summary_view = summary_df.reindex(columns=summary_cols).copy()

if not summary_view.empty:
    total_runs_nonzero = summary_view["total_runs"].replace({0: pd.NA})
    summary_view["completion_rate_pct"] = (summary_view["completed_runs"] / total_runs_nonzero * 100).round(1)
    summary_view["avg_duration_s"] = (summary_view["average_duration_ms"] / 1000).round(2)

summary_view.sort_values("completion_rate_pct", ascending=False)

In [ ]:
if not summary_view.empty and summary_view["completion_rate_pct"].notna().any():
    plot_df = summary_view[["task_name", "completion_rate_pct"]].sort_values("completion_rate_pct")
    ax = plot_df.plot(kind="barh", x="task_name", y="completion_rate_pct", legend=False, figsize=(8, 4))
    ax.set_xlabel("Completion rate (%)")
    ax.set_ylabel("")
    ax.set_title("Completion rate by task")
    plt.tight_layout()
else:
    print("No summary data available for plotting.")

## Run-level diagnostics

In [ ]:
run_cols = [
    "task_name",
    "run_id",
    "interface_type",
    "completed",
    "duration_ms",
    "validation_attempts",
    "validation_successes",
    "validation_failures",
    "questionnaire_completed",
]

runs_view = runs_df.reindex(columns=run_cols).copy()
if "duration_ms" in runs_view.columns:
    runs_view["duration_s"] = (runs_view["duration_ms"] / 1000).round(2)

runs_view.sort_values(["task_name", "run_id"]).head(20)

In [ ]:
if not runs_df.empty and {"task_name", "run_id", "completed", "duration_ms", "validation_attempts"}.issubset(runs_df.columns):
    agg = runs_df.groupby("task_name", dropna=False).agg(
        runs=("run_id", "count"),
        completed_runs=("completed", "sum"),
        avg_duration_s=("duration_ms", lambda s: round(s.mean() / 1000, 2)),
        median_duration_s=("duration_ms", lambda s: round(s.median() / 1000, 2)),
        avg_validation_attempts=("validation_attempts", "mean"),
    )

    agg["completion_rate_pct"] = (agg["completed_runs"] / agg["runs"] * 100).round(1)
    agg["avg_validation_attempts"] = agg["avg_validation_attempts"].round(2)
    display(agg.sort_values("completion_rate_pct", ascending=False))
else:
    print("Not enough run data to build aggregate diagnostics.")

In [ ]:
if not runs_df.empty and {"duration_ms", "task_name"}.issubset(runs_df.columns):
    runs_df.boxplot(column="duration_ms", by="task_name", figsize=(10, 5), rot=20)
    plt.title("Run duration distribution by task")
    plt.suptitle("")
    plt.ylabel("Duration (ms)")
    plt.tight_layout()
else:
    print("No run duration data available for boxplot.")

## Validation check failure hotspots

In [ ]:
if not logs_df.empty and {"action", "result", "task_name", "target"}.issubset(logs_df.columns):
    validation_checks = logs_df[(logs_df["action"] == "validate_check") & (logs_df["result"] == "failed")].copy()

    if not validation_checks.empty:
        failed_by_target = (
            validation_checks
            .groupby(["task_name", "target"], dropna=False)
            .size()
            .reset_index(name="failed_count")
            .sort_values(["task_name", "failed_count"], ascending=[True, False])
        )
        display(failed_by_target.groupby("task_name", dropna=False).head(10))
    else:
        print("No failed validation checks found.")
else:
    print("No validation logs available for failure hotspot analysis.")

In [ ]:
if not logs_df.empty and {"action", "task_name", "run_id"}.issubset(logs_df.columns):
    attempt_logs = logs_df[logs_df["action"] == "validate_attempt"].copy()

    if not attempt_logs.empty:
        attempts_per_run = (
            attempt_logs
            .groupby(["task_name", "run_id"], dropna=False)
            .size()
            .reset_index(name="attempt_count")
            .sort_values(["task_name", "run_id"])
        )

        display(attempts_per_run.head(30))

        fig, ax = plt.subplots(figsize=(10, 4))
        for task_name, g in attempts_per_run.groupby("task_name"):
            ax.plot(g["run_id"], g["attempt_count"], marker="o", label=task_name)

        ax.set_title("Validation attempts per run")
        ax.set_xlabel("Run ID")
        ax.set_ylabel("Validation attempts")
        ax.legend()
        plt.tight_layout()
    else:
        print("No validation attempt logs found.")
else:
    print("No logs available for attempt trend analysis.")

## User action endpoint activity

In [ ]:
if not logs_df.empty and {"level", "task_name", "target"}.issubset(logs_df.columns):
    user_actions = logs_df[logs_df["level"] == "user_action"].copy()

    if not user_actions.empty:
        endpoint_counts = (
            user_actions
            .groupby(["task_name", "target"], dropna=False)
            .size()
            .reset_index(name="count")
            .sort_values(["task_name", "count"], ascending=[True, False])
        )

        display(endpoint_counts.groupby("task_name", dropna=False).head(15))
    else:
        print("No user_action logs found.")
else:
    print("No logs available for endpoint activity analysis.")

## Quick filters for ad-hoc debugging

In [ ]:
# Example: inspect one task and one run.
if not runs_df.empty and {"task_name", "run_id"}.issubset(runs_df.columns):
    task = runs_df["task_name"].iloc[0]
    run_id = 1

    print(f"Task: {task}, run_id: {run_id}")
    display(runs_df[(runs_df["task_name"] == task) & (runs_df["run_id"] == run_id)])

    if not logs_df.empty and {"task_name", "run_id"}.issubset(logs_df.columns):
        display(logs_df[(logs_df["task_name"] == task) & (logs_df["run_id"] == run_id)].head(50))
else:
    print("No run data available.")